In [ ]:
using COBREXA, GLPK, CPLEX, Tulip, DataFrames, Distributed, CSV
include("map_gene_reaction.jl")

addprocs(12)    
@everywhere using COBREXA, GLPK 
 
MODEL_PATH = "model/Human-GEM.mat" 
model = load_model(MODEL_PATH);
gene_expr_df = CSV.read("data\\normalised_metabric_expression.csv", DataFrame)
genes = gene_expr_df[!,"Column1"]
gene_exp = select(gene_expr_df, Not(:"Column1"))
print(size(gene_exp))
GAMMA = 2

(20376, 1832)

2

In [2]:
gene_exp

Row,MB-0362,MB-0346,MB-0386,MB-0574,MB-0185,MB-0503,MB-0641,MB-0201,MB-0218,MB-0316,MB-0891,MB-0658,MB-0899,MB-0605,MB-0258,MB-0506,MB-0420,MB-0223,MB-0445,MB-0199,MB-0517,MB-0155,MB-0428,MB-0117,MB-0906,MB-0249,MB-0660,MB-0497,MB-0434,MB-0143,MB-0513,MB-0541,MB-0653,MB-0455,MB-0540,MB-0384,MB-0637,MB-0157,MB-0443,MB-0584,MB-0292,MB-0322,MB-0501,MB-0401,MB-0140,MB-0606,MB-0666,MB-0598,MB-0453,MB-0138,MB-0579,MB-0405,MB-0471,MB-0347,MB-0310,MB-0621,MB-0614,MB-0372,MB-0374,MB-0382,MB-0327,MB-0066,MB-0144,MB-0596,MB-0164,MB-0215,MB-0146,MB-0229,MB-0505,MB-0102,MB-0569,MB-0516,MB-0272,MB-0585,MB-0494,MB-0306,MB-0463,MB-0198,MB-0203,MB-0607,MB-0631,MB-0363,MB-0427,MB-0519,MB-0371,MB-0380,MB-0221,MB-0348,MB-0261,MB-0576,MB-0385,MB-0270,MB-0379,MB-0432,MB-0527,MB-0624,MB-0273,MB-0460,MB-0654,MB-0454,⋯
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,⋯
1,0.930484,1.03521,0.968726,0.94527,0.936857,0.994535,0.995856,0.904787,0.919009,0.953553,0.970911,0.889308,0.926914,0.919669,0.980752,0.916765,0.907732,1.01558,0.957819,0.976732,0.996643,1.00931,0.850186,0.995901,1.01494,0.911488,0.966672,0.994224,0.853935,0.906813,0.980929,0.9634,0.913182,0.896406,1.02045,0.968234,0.870463,0.920149,0.955379,0.919842,0.869938,0.919784,0.970864,0.87682,0.90464,0.901691,0.879647,0.88902,0.971685,1.01588,1.00538,1.05369,1.00648,0.969896,0.931708,1.0005,0.952283,0.829749,0.920543,1.02978,0.906106,0.887444,0.977159,0.957326,0.907618,0.918868,0.965247,0.930844,1.00687,0.913584,0.918023,0.941389,0.916068,0.858524,0.910027,0.98367,0.957136,0.927896,0.936172,0.979537,0.990267,0.97816,0.954013,1.04383,0.914398,0.976275,1.01197,0.99594,0.99157,0.962628,0.958222,0.876559,0.966241,0.944326,1.03624,0.983692,0.932504,0.978984,0.999041,1.00375,⋯
2,0.900239,0.991007,0.875872,0.834064,0.947226,0.875548,0.91971,0.903242,0.945908,0.855542,0.902927,0.99652,1.03877,0.897773,0.925835,0.817528,0.872632,0.919269,0.829705,0.974087,0.891351,0.917128,0.833834,1.06978,0.861411,0.981706,0.960767,0.868668,0.957062,0.945716,1.01938,0.829433,0.854327,0.926445,1.01102,0.822698,0.813465,0.871154,0.94183,0.846503,0.883615,0.831646,0.842195,1.08609,0.869466,0.867653,0.824872,0.890716,1.00925,0.913881,1.09893,1.00844,1.01679,0.874909,0.955197,1.03742,0.964441,1.17045,0.792735,1.04393,0.897184,1.0614,0.813477,0.895279,0.885568,0.867016,0.941094,0.869976,1.01148,1.12398,0.796437,0.983832,0.884978,0.865488,0.902784,0.905704,0.848053,1.00083,0.854742,0.948264,0.832575,0.96139,0.997075,0.938821,0.838308,0.812214,0.8184,0.824734,0.872227,0.895915,0.997075,0.9268,0.846142,0.832376,1.09074,0.994248,1.01768,0.872242,0.795934,0.852111,⋯
3,0.980345,0.940482,1.01266,0.982219,0.930619,1.00157,0.971127,0.963436,0.994728,0.991084,0.983168,0.950324,0.988384,1.0133,1.03432,0.929075,0.902187,0.986862,0.98823,0.976607,0.993031,0.989578,0.990718,0.948295,0.966964,0.944237,0.964938,1.05248,0.993217,1.00631,1.17399,1.03989,0.930452,0.957807,0.939096,0.937307,1.00519,0.98148,1.05979,1.01504,0.995861,1.01821,1.01928,0.947285,1.08989,1.05253,0.965702,0.984,0.980915,0.943706,0.968411,1.02558,0.947862,0.957789,0.987206,0.958377,0.950004,0.916322,0.98781,0.97192,1.00483,0.967913,1.04824,0.989243,0.953078,0.934208,1.02737,0.98618,1.04033,0.933281,0.92741,0.90214,0.968147,0.92011,0

In [3]:
bound = bounds(model)

([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0, 1000.0  …  1000.0, 1000.0, 1000.0, 1000.0, 0.288633977201338, 0.00590380298884837, 0.00959405479991621, 0.000901866820618997, 1000.0, 1000.0])

In [ ]:
print("Generating fluxes...")

flux_df_max = Dict{String,Any}()
count = 1;
for i in 1: size(gene_exp)[2]
    colname = string(names(gene_exp)[i]) 
    print('\n', "sample: ", i, colname, '\n')
    values = gene_exp[!,i]
    rxnExpression = mapExpressionsToReactions(model, genes, values)
    const_model = constrainReactions(model, rxnExpression, GAMMA)
    #print(bounds(const_model))
    try
        minFlux, maxFlux = flux_variability_analysis_dict(
            const_model,
            GLPK.Optimizer;
            #Tulip.Optimizer;
            workers=workers(),
            bounds = objective_bounds(0.9),
            #modifications = constrains,
        )
        rxn_ids = keys(maxFlux)
        maxFluxRxn = Dict{String,Vector{Maybe{Float64}}}()
        for rxn_id in rxn_ids
            #print(rxn_id)
            ub = maxFlux[rxn_id][rxn_id]
            maxFluxRxn[rxn_id] = [ub]
        end
        flux_df_max[colname] = maxFluxRxn
        #print(maxFluxRxn)
    catch
        println("error occured")
        println(colname)
        flux_list = []
        for (key, value) in flux_df_max
            df = DataFrame(value)
            df[!, "index"] .= key
            push!(flux_list,df)
        end
        df = vcat(flux_list...)
        print("printing file ...")
        CSV.write("Results\\MaxFlux.csv", df)
    end
    count = count + 1
end
flux_list = []
for (key, value) in flux_df_max
    df = DataFrame(value)
    df[!, "index"] .= key
    push!(flux_list,df)
end
df = vcat(flux_list...)
print("printing file ...")
CSV.write("Results\\MaxFlux.csv", df)

Generating fluxes...
sample: 1MB-0362


Mapping transcriptomics data to reactions 


Finished constraining model with transcriptomics data 



sample: 2MB-0346


Mapping transcriptomics data to reactions 
Finished constraining model with transcriptomics data 


...

Finished constraining model with transcriptomics data 



sample: 987MB-2927


Mapping transcriptomics data to reactions 
Finished constraining model with transcriptomics data 



sample: 988MB-2919
Mapping transcriptomics data to reactions 


Finished constraining model with transcriptomics data 


In [ ]:
df